# Pya - Tour of AGens

- This notebook contains examples and explanations for available AGens provided
with the pya package. 
- For information about the logic around pya, cf. the
Jupyter notebook pya-examples-agen.ipynb. 

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

from pya import Aserver, Asig, startup, device_info
from pya.agen.lib import SinOsc, Line

mpl.rcParams['figure.figsize'] = (8,2.5)

device_info();
s = startup() # optionally add device={idx}
(SinOsc(1000) * Line(0.1, 0, 0.1)).play();

## Oscillators

### LFSaw(freq, phase)

This is a Sawtooth Oscillator: 
- non-band-limited, 
- value range is [-1, 1], 
- The frequency (GenOrNum) is given in Hz 
- it starts at zero with positive slope.
- the phase (GenorNum) argument is a normalized phase, i.e. range [0,1] (instead of 0, 2pi)

In [ ]:
from pya.agen.lib import LFSaw
from pya.agen.core import stereo, multi_channel
LFSaw(50, multi_channel(0, 0.25, 0.5, 0.75, 1)).gen_asig(seconds=0.1).plot(offset=3)

In [ ]:
# can be used for audio creation, but mind the aliasing
(LFSaw(50) * Line(0.1, 0, 2)).play();

In [ ]:
# often used for control signal, e.g. here to control a BLImp to create an alert
from pya.agen.lib import BLImp
agfreq = LFSaw(2, 0.5).linlin(-1, 1, 200, 300)
(BLImp(agfreq, 8) * Line(0.2, 1, 2)).play();

Here an example where the phase is modulated to detune two LFSaws

In [ ]:
from pya.agen.lib import Env
agenv = Env([0,0.2,0.2,0], [0.5,3,0.5])
(LFSaw(stereo(70, 70), stereo(0, SinOsc(0.4).linlin(-1,1,0,0.25))).mix() * agenv).play();

### LFTri(freq, phase)

This is a Triangle Oscillator: 
- non-band-limited, 
- value range is [-1, 1], 
- The frequency (GenOrNum) is given in Hz 
- it starts at zero with positive slope.
- the phase (GenorNum) argument is a normalized phase, i.e. range [0, 1] (instead of 0, 2pi)
- Technically, a triangle is abs(LFSaw), scaled and shifted to target range

In [ ]:
from pya.agen.lib import LFTri
from pya.agen.core import stereo, multi_channel
LFTri(50, multi_channel(0, 0.25, 0.5, 0.75, 1)).gen_asig(seconds=0.1).plot(offset=3)

- LFTri can be used for audio creation, but mind the aliasing
- The example plays a set of Triangle tones interleaved with SinOsc tones

In [ ]:
for i, f in enumerate([110, 220, 440, 880, 1760, 3520]):
    (LFTri(f, 0) * Line(0.1, 0, 1)).play(onset=i)
    (SinOsc(f, 0) * Line(0.1, 0, 1)).play(onset=i + 0.5)

In [ ]:
# often used for control signal, e.g. here to control a BLImp to create a joddling
from pya.agen.lib import Env 

agfreq = (LFTri(5) * Line(0.3, 1, 2)).linlin(-1, 1, 0, 80)
(SinOsc(400, phase=agfreq) * Env([0, 0.2, 0.2,0], [0.2, 2, 0.2]) ).play();

- an interesting use of LFTri is as phasor into a buffer, -> AsigRead